In [8]:
import torch
import torch.nn as nn

class Mymodel(nn.Module): # nn.Module -> 신경망 블록의 기본 틀
    def __init__(self):  # __init__ 레이터/파라미터를 준비하는 곳
        super().__init__() #부모 클래스인 nn.Module의 초기화 함수도 실행

        self.linear = nn.Linear(4, 2) #이 모델 안에 입력 4차원 → 출력 2차원으로 바꾸는 Linear layer 하나를 넣겠다

    def forward(self,x): #forward - > 실제 입력이 들어왔을 때 계산하는 곳

        x = self.linear(x)

        return x
#model(x) → nn.Module.__call__() → forward(x)


In [9]:
#forward() 입력이 모델 안으로 들어왔을 때 실제로 어떤 계산을 할리 적는 함수
# model(x) - >nn.Module.__call__(x) - > forward(x)  -> 결과 return
#nn.Linear은 쉽게 말하면 입력 벡터에 가중치 행렬을 곱하고 bias를 더해서 다른 차원의 벡터로 바꾸는 층

x = torch.randn(4, 20, 128)

linear = nn.Linear(128, 64)

output = linear(x)

print(output.shape)


torch.Size([4, 20, 64])


In [10]:
#parameters() nn.module안에서 학습되는 파라미터를 꺼내보는 함수

class MyModel(nn.Module):
   def __init__(self):
     super().__init__()
     self.linear = nn.Linear(4,2)

   def forward(self ,x):
      return self.linear(x)

model = MyModel()

for p in model.parameters():
    print(p.shape) #weight shape [2,4],bias 출력 차원 만큼있음


torch.Size([2, 4])
torch.Size([2])


In [11]:
#named_modules()는 모델 안에 들어있는 모든 nn.Module들을 이름과 함께 보여주는 함수

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(4,8)
        self.relu = nn.ReLU()
        self.linear2 =nn.Linear(8,2)


    def forward(self,x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x
model = MyModel()

for name,module in model.named_modules():
   print(name,module)

 MyModel(
  (linear1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=8, out_features=2, bias=True)
)
linear1 Linear(in_features=4, out_features=8, bias=True)
relu ReLU()
linear2 Linear(in_features=8, out_features=2, bias=True)


In [7]:
#forward hook : 어떤 nn.module의 forward가 실행된 직후에 자동으로 끼어들어서 입력/출력을 볼 수 있게 하는 함수
def hook_fn(module,inputs,output):
  print("module :", module)
  print("input shape:", inputs[0].shape)
  print("output shape :",output.shape)

  model = MyModel()
  handle = model.linear.register_forward_hook(hook_fn)
 #handle.remove()
  x = torch.randn(3, 4)
  output = model(x)


In [18]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear1 =nn.Linear(16,32)
        self.relu = nn.ReLU()
        self.linear2 =nn.Linear(32,8)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x


def hook_fn(module, inputs, output):
      print("module:", module)
      print("input shape:", inputs[0].shape)
      print("output shape:", output.shape)

model = MyModel()

x = torch.randn(5, 16)

handle = model.linear1.register_forward_hook(hook_fn)

output = model(x)


print(output.shape)

for p in model.parameters():
  print(p.shape)

for name,module in model.named_modules():
  print(name,module)

module: Linear(in_features=16, out_features=32, bias=True)
input shape: torch.Size([5, 16])
output shape: torch.Size([5, 32])
torch.Size([5, 8])
torch.Size([32, 16])
torch.Size([32])
torch.Size([8, 32])
torch.Size([8])
 MyModel(
  (linear1): Linear(in_features=16, out_features=32, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=32, out_features=8, bias=True)
)
linear1 Linear(in_features=16, out_features=32, bias=True)
relu ReLU()
linear2 Linear(in_features=32, out_features=8, bias=True)


In [22]:
class SimpleAttention(nn.Module):
    def __init__(self, d_model,num_heads):
        super().__init__()

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

    def forward(self, x):

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        B,T,D = q.shape
        q = q.reshape(B,T,self.num_heads, self.head_dim)
        k = k.reshape(B,T,self.num_heads, self.head_dim)
        v = v.reshape(B,T,self.num_heads, self.head_dim)

        q = q.transpose(1,2)
        k = k.transpose(1,2)
        v = v.transpose(1,2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        attention_output = weights @ v
        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.reshape(B, T, D)
        return q, k, v,attention_output


model = SimpleAttention(128,8)

x = torch.randn(3, 20, 128)

q, k, v, attention_output = model(x)

print(q.shape)
print(k.shape)
print(v.shape)
print(attention_output.shape)

torch.Size([3, 8, 20, 16])
torch.Size([3, 8, 20, 16])
torch.Size([3, 8, 20, 16])
torch.Size([3, 20, 128])
